In [0]:
spark.version

'4.1.0'

In [0]:
pip install duckdb pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


In [0]:
from pyspark.sql.functions import col, expr, max, count, min, sum, avg,year, month, dense_rank, desc, lag, countDistinct
from pyspark.sql.window    import Window
from pyspark.sql.types     import StructType, StructField, StringType, IntegerType, DateType


In [0]:
raw_cust_df = ( spark.read.format('csv')
                          .option("header", True)
                          .option("inferSchema", True)
                          .load("/Volumes/dev/spark_db/datasets/spark_programmingHBL/data/customers.csv")
              )

#raw_cust_df.display()

raw_ord_df = ( spark.read.format('csv')
                          .option("header", True)
                          .option("inferSchema", True)
                          .load("/Volumes/dev/spark_db/datasets/spark_programmingHBL/data/orders.csv")
              )


raw_prod_df = ( spark.read.format('csv')
                          .option("header", True)
                          .option("inferSchema", True)
                          .load("/Volumes/dev/spark_db/datasets/spark_programmingHBL/data/products.csv")
              )


raw_ret_df = ( spark.read.format('csv')
                          .option("header", True)
                          .option("inferSchema", True)
                          .load("/Volumes/dev/spark_db/datasets/spark_programmingHBL/data/returns.csv")
              )


raw_emp_df = ( spark.read.format('csv')
                          .option("header", True)
                          .option("inferSchema", True)
                          .load("/Volumes/dev/spark_db/datasets/spark_programming/data/employee.csv")
              )

raw_dep_df = ( spark.read.format('csv')
                         .option("header", True)
                         .option("inferSchema", True)
                         .load("/Volumes/dev/spark_db/datasets/spark_programming/data/department.csv")
              )



In [0]:
# Option One reading csv file from Volume

#raw_cust_df.display()
#raw_ord_df.display()
#raw_prod_df.display()
#raw_ret_df.display()

#raw_emp_df.display()

#raw_dep_df.display()





### TBD




### Practices

#### Query

In [0]:
%sql

SELECT 
FROM  dev.spark_db.department as d

;



In [0]:
from pyspark.sql.functions import col, expr, max, count, min, sum, avg,year, month, dense_rank, desc, lag, countDistinct, date_diff, lit
from pyspark.sql.window    import Window
from pyspark.sql.types     import StructType, StructField, StringType, IntegerType, DateType

In [0]:

grouped_date = (raw_cust_df.groupBy("order_date", "product_name")
                           .agg( expr('sum(quantity * price) as Sales') )
               )

windowSpec= Window.partitionBy("product_name").orderBy("order_date")

cust_df = ( grouped_date.withColumns({"Prev_sales": lag('Sales').over(windowSpec),
                                      "Sales_diff": expr('Sales - Prev_sales')
                                    })
                        .orderBy("product_name", "order_date")

         )

cust_df.display()       

order_date,product_name,Sales,Prev_sales,Sales_diff
2019-11-17,Doodad,20,null,null
2020-04-01,Doodad,16,20,-4
2026-02-17,Doodad,80,16,64
2026-03-11,Doodad,160,80,80
2020-01-12,Gadget,150,null,null
2024-01-01,Gadget,150,150,0
2024-02-01,Gadget,300,150,150
2024-03-01,Gadget,300,300,0
2024-04-01,Gadget,300,300,0
2024-05-01,Gadget,300,300,0


In [0]:
%sql
-- Monthly sales revenue and order count

SELECT extract(MONTH FROM ORDER_DATE) as MTH, SUM(QUANTITY * PRICE) AS SALES, COUNT(DISTINCT ORDER_ID) AS COUNT
FROM   dev.spark_db.customers as c
GROUP BY extract(MONTH FROM ORDER_DATE)
order by MTH
;




### Standard way

In [0]:
%sql
from pyspark.sql.functions import col, expr, count

cust_df = ( raw_cust_df.groupBy('month(order_date)')
                       .agg(expr('sum(quantity * price) as SALES'),
                            expr('count(order_id) as CNT')
                           )
                        
          )

cust_df.display()


### Temporary view

In [0]:
from pyspark.sql.functions import col, expr, count
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

raw_emp_df.createOrReplaceTempView("employee")

sql_query = ( """
              WITH T1 AS ( SELECT SALARY , DENSE_RANK() OVER( ORDER BY SALARY DESC) AS RN
                          FROM dev.spark_db.employee
                         GROUP BY SALARY   

                         )

              SELECT  SALARY
              FROM T1
              WHERE RN = 2;
              """
            )
result_df = spark.sql(sql_query)
result_df.display()          


#### What San Francisco neighborhoods in in the zip codes 94102 and 94103

In [0]:
%sql
-- City, Neighborhood, Zipcode

SELECT City, Neighborhood, Zipcode
FROM dev.spark_db.sf_fire_calls
WHERE City = 'SF' and Zipcode in (94102 , 94103);

In [0]:

from pyspark.sql.window    import Window
from pyspark.sql.functions import rank, col, count, sum, expr, desc

result_df = ( raw_fire_df.where( (raw_fire_df["Zipcode"].isin([94102 , 94103])) & (raw_fire_df["City"]=='SF') )
                         .select("City", "Neighborhood", "Zipcode")
            )

result_df.display()